Prerequisites
Ensure you have the transformers and torch libraries installed:

In [1]:
pip install transformers torch

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class PythonCodeAssistant:
    def __init__(self, model_name="gpt2"):
        print(f"Loading {model_name}...")
        # 1. Load Model and Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)

        # Set pad token to eos token to avoid warnings (GPT-2 doesn't have a pad token)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        print("Model loaded successfully.")

    def generate_response(self, user_prompt):
        """
        Handles the logic for filtering and generating responses.
        """

        # 2. Implement a Filtering Mechanism (Using Few-Shot Prompting)
        # We provide examples (shots) so GPT-2 understands the pattern:
        # - If it's code -> Provide code.
        # - If it's not code -> Say "NOT_PYTHON".

        few_shot_prompt = """
Task: You are a Python coding assistant. Answer only Python questions. If the question is not about Python, say "NOT_PYTHON".

Q: What is the capital of France?
A: NOT_PYTHON

Q: How do I define a function in Python?
A: def my_function(): pass

Q: Who won the world cup?
A: NOT_PYTHON

Q: Write a loop to print numbers 1 to 10.
A: for i in range(1, 11): print(i)

Q: What is the weather like?
A: NOT_PYTHON

Q: {}
A:""".format(user_prompt)

        # Encode inputs
        inputs = self.tokenizer(few_shot_prompt, return_tensors="pt")

        # 3. Generate Response
        # We limit max_new_tokens to prevent it from rambling
        with torch.no_grad():
            outputs = self.model.generate(
                inputs.input_ids,
                max_new_tokens=50,
                pad_token_id=self.tokenizer.eos_token_id,
                temperature=0.7, # Lower temperature for more deterministic answers
                do_sample=True
            )

        # Decode the full text
        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract only the new part (the actual answer)
        # We split by the prompt we sent to find the "A:" part
        answer = full_text[len(few_shot_prompt):].strip()

        # 4. Handle Non-Coding Questions & Post-Processing
        # If the model followed our few-shot pattern, it should output "NOT_PYTHON" for non-code.
        if "NOT_PYTHON" in answer:
            return "I can only answer questions related to Python coding."

        return answer

# --- 5. Test Implementation ---
if __name__ == "__main__":
    bot = PythonCodeAssistant()

    test_questions = [
        "How do I print 'Hello World' in Python?",  # Coding
        "What is the best movie of 2020?",          # Non-Coding
        "How do I import pandas?",                  # Coding
        "Who is the president of USA?"              # Non-Coding
    ]

    print("-" * 50)
    for q in test_questions:
        print(f"User: {q}")
        response = bot.generate_response(q)
        print(f"Bot:  {response}\n")
        print("-" * 50)

Loading gpt2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model loaded successfully.
--------------------------------------------------
User: How do I print 'Hello World' in Python?
Bot:  I can only answer questions related to Python coding.

--------------------------------------------------
User: What is the best movie of 2020?
Bot:  Not only Arendt's is a movie which will be a huge success. It's a great movie for young kids. It is one of the best movies I've seen in my life.

Q: How do you use the help book

--------------------------------------------------
User: How do I import pandas?
Bot:  import pandas as pd

Q: How do I create a web service?

A: my_service = pd.service( "my_service.py" )

Q: How do I run my web service

--------------------------------------------------
User: Who is the president of USA?
Bot:  I can only answer questions related to Python coding.

--------------------------------------------------
